In [ ]:
CONFIG = {
    # Dataset
    "DATA_PATH": "path/to/full_state_data.csv", # [T, N_Buses, Features(P,Q,V,d)]
    
    # Splitting (Interleaved 4:1:1 Strategy)
    "BLOCK_SIZE_HOURS": 168,                    # 1 Woche
    "CYCLE_SCHEME": [0, 1, 2, 3, 4, 5],         # 0-3=Train, 4=Val, 5=Test
    
    # Forecasting Task (Triebe Style)
    "INPUT_WINDOW": 168,                        # Lookback: 7 Tage
    "FORECAST_HORIZON": 33,                     # Predict: Rest of Day (9h) + Next Day (24h)
    "TARGET_DAY_START": 9,                      # Ab Stunde 9 beginnt der "Target Day" (00:00 morgen)
    "EVAL_HOUR": 14,                            # Simulation: Wir predicten immer um 14:00 Uhr
    
    # Baselines
    "SNAIVE_LAG": 48,                           # Vergleichswert: Vorgestern zur gleichen Zeit
    
    # Features
    "USE_FULL_STATE": True                      # TGT/XGBoost nutzen P,Q,V,Delta
}

In [3]:
import math, numpy as np, numpy.linalg as LA, networkx as nx, matplotlib.pyplot as plt, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

ModuleNotFoundError: No module named 'torch'

In [ ]:
#! Set working dir
import os
print(os.getcwd())
os.chdir("../")
print(os.getcwd())

In [ ]:
# Load Bus and Branch DataFrames

bus_df = pd.read_parquet('data_out/no_pertubations/case14_ieee/raw/bus_data.parquet')
branch_df = pd.read_parquet('data_out/no_pertubations/case14_ieee/raw/branch_data.parquet')

# Build Edge Index from Branch Data
structure = branch_df[branch_df['load_scenario_idx'] == 0].sort_values("idx")
edge_index = list(structure[['from_bus', 'to_bus']].itertuples(index=False, name=None))

# print dims
m = len(edge_index)
n=bus_df['bus'].nunique()
print(f"Topology Loaded: n={n} Nodes, m={m} Edges")

# X: Input Loads (Active Power 'Pd')
loads_pivot = bus_df.pivot(index='load_scenario_idx', columns='bus', values='Pd').fillna(0)
# Y: Target Flows (Active Power Flow 'pf')
flows_pivot = branch_df.pivot(index='load_scenario_idx', columns='idx', values='pf').fillna(0)

# Convert to Numpy Arrays
loads_matrix = loads_pivot.values.astype(np.float32)
flows_matrix = flows_pivot.values.astype(np.float32)
print(f"shapes: Loads: {loads_matrix.shape}, Flows: {flows_matrix.shape}")

Extract following data from Bus_df:
Pd	bus_data	Active Load (Das Ziel)
Qd	bus_data	Reactive Load
Pg	bus_data	Active Generation
Qg	bus_data	Reactive Generation
Vm	bus_data	Voltage Magnitude
Va	bus_data	Voltage Angle
Sin/Cos	(Calculated)	6x Time Features

In [ ]:

# Pseudo-Code zum Laden der Features
def load_rich_features(path):
    # Lade bus_data (enthält Pd, Qd, Pg, Qg, Vm, Va)
    df = pd.read_parquet(path + "/bus_data.parquet")
    
    # WICHTIG: Sortiere sicherheitshalber nach Zeit und Bus ID
    df = df.sort_values(["scenario", "bus"])
    
    # Extrahiere die Spalten als Numpy Tensor [Time, Nodes, Features]
    # Features: Pd, Qd, Pg, Qg, Vm, Va
    features = ['Pd', 'Qd', 'Pg', 'Qg', 'Vm', 'Va']
    tensor = df[features].values.reshape(n_timesteps, n_buses, 6)
    
    return tensor

In [ ]:
import numpy as np
import pandas as pd

def add_cyclical_features(df):
    """
    Encodes Hour, DayOfWeek, DayOfYear into Sin/Cos pairs.
    Total new features: 6
    """
    # 1. Hour of Day (0-23)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)

    # 2. Day of Week (0-6)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

    # 3. Day of Year (0-365)
    # Note: Use 366.0 for leap year compatibility or 365.25 standard
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

    return df

import numpy as np

def get_interleaved_splits(total_hours, block_size=168, input_window=168):
    """
    Implements the 4:1:1 Golden Cycle Strategy.
    Cycle: Train -> Train -> Train -> Train -> Val -> Test
    
    Args:
        total_hours: Total length of dataset (e.g., 8760 for 1 year)
        block_size: Size of one chunk (168 hours = 1 week)
        input_window: How much history the model needs (Lookback)
    
    Returns:
        train_idx, val_idx, test_idx (arrays of time steps)
    """
    n_blocks = total_hours // block_size
    
    train_indices = []
    val_indices = []
    test_indices = []
    
    # Der 6-Wochen-Zyklus
    # 0,1,2,3 -> Train
    # 4       -> Val
    # 5       -> Test
    
    for i in range(n_blocks):
        # Berechne Start- und Endstunde dieses Blocks
        start_t = i * block_size
        end_t = start_t + block_size
        
        # Determine position in the 6-week cycle
        cycle_pos = i % 6
        
        # Erstelle die Range für diesen Block
        # WICHTIG: Wir nehmen alles von start bis ende
        block_range = np.arange(start_t, end_t)
        
        if cycle_pos in [0, 1, 2, 3]:
            # --- TRAINING ---
            train_indices.extend(block_range)
            
        elif cycle_pos == 4:
            # --- VALIDATION ---
            # Check: Haben wir genug History davor? (Ja, wenn i > 0)
            if start_t >= input_window:
                val_indices.extend(block_range)
            else:
                # Fallback: Wenn der allererste Block Val wäre (passiert hier nicht),
                # pack ihn zu Train, sonst Crash.
                train_indices.extend(block_range)
                
        elif cycle_pos == 5:
            # --- TEST ---
            if start_t >= input_window:
                test_indices.extend(block_range)
            else:
                train_indices.extend(block_range)

    # Konvertiere zu numpy arrays
    return np.array(train_indices), np.array(val_indices), np.array(test_indices)

# --- EXAMPLE USAGE ---
# total_len = 8760 # 1 Year
# train_idx, val_idx, test_idx = get_interleaved_splits(8760)

# print(f"Train Hours: {len(train_idx)}")
# print(f"Val Hours:   {len(val_idx)}")
# print(f"Test Hours:  {len(test_idx)}")

In [2]:
#! Do we even need?
# 4. Scaling (Strict Separation)

#     Fit StandardScaler ONLY on data[train_idx].

#     Transform the entire dataset.

#     Note: You might need separate scalers for P, Q, V, and Delta to invert them correctly later.

C. Rolling Dataset Class (Für Training)

Erstelle eine PyTorch Dataset Klasse für das Training von TGT (und Vorbereitung für XGBoost).

    Logic: __getitem__(index)

        Input X: data[index : index + 168]

        Target Y: data[index + 168 : index + 168 + 33]

    Step Size: 1 Stunde. (Nutze jede mögliche Stunde im Train-Set, um Daten zu maximieren).

# 5. Model Training Pipeline (The Comparison)

A. Baseline 1: Local sNaïve

    Input: None (Just history).

    Logic: Pred[t] = Actual[t - lag variable from config]. 

B. Baseline 2: Local sARIMA

    Input: Univariate P history per bus.

    Tool: pmdarima.auto_arima(..., m=24). (Use m=24 for speed, m=168 is too slow).

C. Baseline 3: Global XGBoost (Two Variants)

    Variant C1: XGBoost-Lite (P-Only)

        Features: [Lags_P, Time_Feat, Bus_ID].

    Variant C2: XGBoost-Pro (Full-State)

        Features: [Lags_P, Lags_Q, Lags_V, Lags_Delta, Time_Feat, Bus_ID].

        Why: Tests if XGBoost can utilize physics correlations without graph structure.

    Implementation: Stack all buses into one large matrix. Use MultiOutputRegressor for output (size defined in config).

D. The Challenger: TGT / GridFM (Two Variants)

    Variant D1: TGT-Lite (Topology + P)

        Input: P history + Adjacency Matrix A.
        Outpout: linear layer, forecast horizon x node (x batch i guess?)
        Hypothesis: Beats XGBoost-Lite due to Graph Structure (A).

    Variant D2: TGT-Pro (Topology + Physics)

        Input: Full state (P,Q,V,δ) + Adjacency Matrix A.

        Hypothesis: Beats TGT-Lite due to implicit physics learning (Voltage sensitivity).

    Training: Standard PyTorch loop. Loss = MSE.

    Trainiere auf RollingDataset(train_idx).

    Validiere nach jeder Epoch auf RollingDataset(val_idx).

    Early Stopping: Wenn Val-Loss 5 Epochen nicht fällt → Stop. Speichere best_model.pth.



def evaluate_daily_2pm(model, full_data, test_indices):
    results = []
    
    # Finde alle Zeitpunkte im Test-Set, die "14:00 Uhr" entsprechen
    test_starts_2pm = [t for t in test_indices if is_2pm(t)]
    
    for t in test_starts_2pm:
        # 1. Input holen (letzte 7 Tage bis heute 14:00)
        x_input = full_data[t - 168 : t] 
        
        # 2. Prediction machen (33h in die Zukunft)
        # Forecast Zeitraum: Heute 14:00 bis Morgen 23:00
        y_pred_33h = model.predict(x_input) 
        
        # 3. Ground Truth holen
        y_true_33h = full_data[t : t + 33]
        
        # 4. SLICING (Nur den "Target Day" bewerten)
        # Wir ignorieren die ersten 9h (Rest von heute)
        # Wir nehmen Stunde 9 bis 33 (00:00 bis 23:59 morgen)
        y_pred_target = y_pred_33h[9:33]
        y_true_target = y_true_33h[9:33]
        
        # 5. Inverse Transform (Zurück zu MW für Metriken!)
        y_pred_mw = scaler.inverse_transform(y_pred_target)
        y_true_mw = scaler.inverse_transform(y_true_target)
        
        results.append((y_pred_mw, y_true_mw))
        
    return results

6. Metrics & Reporting (metrics.py)Berechne die finale Tabelle basierend auf den results aus Schritt 5.Berechne Scaling Factor (Denominator):Laufe einmal über das gesamte Training Set.Berechne MAE des sNaive(lag=48).Das ist dein scale_mae.Berechne Metriken (Test Set):RMSE: $\sqrt{\text{mean}((y - \hat{y})^2)}$MAE: $\text{mean}(|y - \hat{y}|)$MASE: $\frac{\text{MAE (Model)}}{\text{scale\_mae}}$ (Ist der Wert < 1, bist du besser als Naive).MSSE: $\frac{\text{MSE (Model)}}{\text{scale\_mse}}$Visualisierung:Erstelle eine Tabelle: Model | RMSE | MAE | MASE.Plot: Eine "2 PM Prediction" Kurve (24h) vs. Ground Truth für einen volatilen Bus.

D. Plots

    Forecast Plot: 3-Day horizon for a specific bus (choose one with high volatility).

    Error Distribution: Boxplot of MASE per bus. (Shows if TGT improves the "worst" buses).